# Support Vector Machine (SVM)

In [1]:
from dependencies import *
from data_manipulation.data_pipeline import load_pipeline, FEATURES

X_train_bal, y_train_bal, X_test_scaled, y_test, class_weight_dict, FEATURES, scaler = load_pipeline()


Loaded: (5569, 18)
Nulls before imputation:
pl_orbper       260
pl_orbsmax     2237
pl_orbeccen    3315
pl_bmasse      2986
pl_rade        1373
pl_eqt         2402
st_teff         290
st_rad          321
st_mass         206
st_lum          675
st_met          424
st_logg         351
st_age         1246
P_GRAVITY         7
P_DENSITY         7
P_ESCAPE          7
sy_dist         118
dtype: int64

[Tier 2a] pl_orbeccen: filled 3315 with 0.0
[Tier 2b] pl_orbsmax: filled 2203 via Kepler's 3rd law
[Tier 2c] pl_orbper: filled 242 via Kepler's 3rd law (inverse)
[Tier 2d] st_lum: filled 362 via Stefan-Boltzmann law
[Tier 2e] pl_bmasse: filled 2972 via mass-radius relation
[Tier 2f] pl_rade: filled 1359 via inverse mass-radius relation
[Tier 2g] pl_eqt: filled 2073 via equilibrium temp formula
[Tier 2h-j] P_GRAVITY/DENSITY/ESCAPE: filled 0 via physics formulas

Nulls after Tier 2:
pl_orbper        18
pl_orbsmax       34
pl_orbeccen       0
pl_bmasse        14
pl_rade          14
pl_eqt          

/Users/ahteshamalvi/PersonalProjects/exoplanet_ML_classification/.venv/lib/python3.13/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


[Tier 4] RF Iterative Imputer on planetary features: 430 → 0 nulls
[Tier 5a] st_met: filled 424 via grouped median by spectral type
[Tier 5b] st_age: filled 1246 via RF regression + median fallback
[Tier 6] KNN imputer: 118 → 0 nulls

Train shape: (4455, 17)
Test shape:  (1114, 17)

Train distribution (BEFORE balancing):
P_HABITABLE
0.0    4399
1.0      23
2.0      33
Name: count, dtype: int64

Train distribution (AFTER balancing):
P_HABITABLE
0.0    299
1.0    200
2.0    199
Name: count, dtype: int64
Balanced train shape: (698, 17)
Class weights: {np.int64(0): np.float64(0.778149386845039), np.int64(1): np.float64(1.1633333333333333), np.int64(2): np.float64(1.169179229480737)}

Outputs ready for modeling:
  X_train_bal       → (698, 17) (balanced, scaled)
  y_train_bal       → 698 samples
  X_test_scaled     → (1114, 17) (held-out, scaled)
  y_test            → 1114 samples (original distribution)
  class_weight_dict → for class_weight param in classifiers


## Training

In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 1 — TUNE C WITH CROSS-VALIDATION
# ══════════════════════════════════════════════════════════════════════════════

param_grid = {
    "C": [0.01, 0.1, 1, 10, 100]
}

grid = GridSearchCV(
    SVC(
        kernel="linear",
        class_weight=class_weight_dict,
        probability=True,
        random_state=42,
    ),
    param_grid,
    cv=5,
    scoring="f1_macro",   # or "f1_macro" if imbalance matters more
    n_jobs=-1
)

grid.fit(X_train_bal, y_train_bal)

best_C = grid.best_params_["C"]

# ══════════════════════════════════════════════════════════════════════════════
# STEP 2 — USE BEST MODEL (already refit by GridSearchCV)
# ══════════════════════════════════════════════════════════════════════════════
svm_l = grid.best_estimator_

y_pred_svm = svm_l.predict(X_test_scaled)
y_prob_svm = svm_l.predict_proba(X_test_scaled)
y_dec_svm  = svm_l.decision_function(X_test_scaled)

# ══════════════════════════════════════════════════════════════════════════════
# STEP 4 — DIAGNOSTICS (UNCHANGED + EXTRA INSIGHT)
# ══════════════════════════════════════════════════════════════════════════════
class_names = ["Non-Habitable", "Mesoplanet", "Psychroplanet"]
classes = np.array([0.0, 1.0, 2.0])
colors = ["#2196F3", "#FF9800", "#4CAF50"]

print("Linear SVM training complete.")
print(f"Best C used: {best_C}")
print(f"Support vectors per class: {svm_l.n_support_}")
print(f"Total support vectors: {svm_l.n_support_.sum()}/{X_train_bal.shape[0]} training samples")

Linear SVM training complete.
Best C used: 100
Support vectors per class: [25 28 33]
Total support vectors: 86/698 training samples


## Modeling
### DIAGNOSTIC PLOT 1: 

### DIAGNOSTIC PLOT 2:

### DIAGNOSTIC PLOT 3: 

## Metrics
### Classification Report + F1 Scores

### Confusion Matrix (Counts + Normalized)

## Interpretation:
